In [3]:
import sqlite3
import random
import json
import os

# Path to your MIMIC-III SQLite database
DB_PATH = "../Assignment_MIMIC_SQL/database.db"
OUTPUT_DIR = "./generated_prompts"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Connect to the database
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Fetch a random clinical note
def get_random_note():
    cursor.execute("SELECT TEXT FROM NOTEEVENTS WHERE LENGTH(TEXT) > 100 LIMIT 1000")
    notes = cursor.fetchall()
    return random.choice(notes)[0] if notes else None

# Prompt templates
def zero_shot_prompt(note):
    return f"""
You are a medical assistant. Simplify the following clinical note into plain English so that a patient with no medical training can easily understand it.

Clinical Note:
\"\"\"{note}\"\"\"

Simplified Version:
"""

def few_shot_prompt(note):
    return f"""
You are a medical assistant. Simplify clinical notes into plain English for patients.

Example 1:
Clinical Note: "The patient was diagnosed with pneumonia based on chest X-ray showing lung infiltrates."
Simplified Version: "You have a lung infection called pneumonia. It was found on your chest X-ray."

Example 2:
Clinical Note: "The patient is experiencing hyperglycemia secondary to poorly controlled diabetes mellitus type II."
Simplified Version: "Your blood sugar is too high because your diabetes hasn't been well controlled."

Now simplify this note:

Clinical Note:
\"\"\"{note}\"\"\"

Simplified Version:
"""

def chain_of_thought_prompt(note):
    return f"""
You are a medical assistant. Simplify clinical notes into plain English for patients.  
First, list the key medical terms and their meanings. Then rewrite the note.

Clinical Note:
\"\"\"{note}\"\"\"

Step 1: Identify terms and meanings.
- Term 1: Meaning
- Term 2: Meaning
- ...

Step 2: Simplified Version:
"""

def reasoning_and_diagnosis_prompt(note):
    return f"""
You are a medical assistant AI. Given a clinical note, identify the most likely diagnosis and explain your reasoning.  

Clinical Note:
\"\"\"{note}\"\"\"

Your Reasoning:
1. Analyze the symptoms and context.
2. Match symptoms to a likely diagnosis.
3. Output the diagnosis and rationale.

Most likely diagnosis:
"""

# Generate and export prompts to a JSON file
def export_prompts(note_id, note_text):
    prompts = {
        "note_id": note_id,
        "original_note": note_text,
        "zero_shot": zero_shot_prompt(note_text),
        "few_shot": few_shot_prompt(note_text),
        "chain_of_thought": chain_of_thought_prompt(note_text),
        "reasoning_and_diagnosis": reasoning_and_diagnosis_prompt(note_text)
    }
    
    filename = os.path.join(OUTPUT_DIR, f"note_{note_id}.json")
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(prompts, f, indent=4)
    print(f"✅ Prompts saved to: {filename}")

# Pull a note and generate prompts
cursor.execute("SELECT ROWID, TEXT FROM NOTEEVENTS WHERE LENGTH(TEXT) > 100 LIMIT 1000")
notes = cursor.fetchall()

if notes:
    random_note = random.choice(notes)
    note_id, note_text = random_note[0], random_note[1]
    export_prompts(note_id, note_text)
else:
    print("No clinical notes found in the database!")

# Close connection
conn.close()


✅ Prompts saved to: ./generated_prompts/note_64.json


In [4]:
# After generating the prompt dictionary...
def export_prompts(note_id, note_text):
    prompts = {
        "note_id": note_id,
        "original_note": note_text,
        "zero_shot": zero_shot_prompt(note_text),
        "few_shot": few_shot_prompt(note_text),
        "chain_of_thought": chain_of_thought_prompt(note_text),
        "reasoning_and_diagnosis": reasoning_and_diagnosis_prompt(note_text)
    }
    
    # 💡 Print a quick preview in console:
    print("\n===================================")
    print(f"📝 Note ID: {note_id}")
    print("🔹 Zero-Shot Prompt:\n", prompts["zero_shot"][:500] + "...\n")
    print("🔹 Few-Shot Prompt:\n", prompts["few_shot"][:500] + "...\n")
    print("🔹 Chain-of-Thought Prompt:\n", prompts["chain_of_thought"][:500] + "...\n")
    print("🔹 Reasoning & Diagnosis Prompt:\n", prompts["reasoning_and_diagnosis"][:500] + "...\n")
    print("===================================\n")

    # Save to JSON
    filename = os.path.join(OUTPUT_DIR, f"note_{note_id}.json")
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(prompts, f, indent=4)
    print(f"✅ Prompts saved to: {filename}")


In [5]:
import sqlite3
import random
import ollama

# Connect to your MIMIC-III DB
DB_PATH = "../Assignment_MIMIC_SQL/database.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Pick a random note
cursor.execute("SELECT ROWID, TEXT FROM NOTEEVENTS WHERE LENGTH(TEXT) > 100 LIMIT 1000")
notes = cursor.fetchall()
note_id, note_text = random.choice(notes)

# Build your Few-Shot Prompt
def few_shot_prompt(note):
    return f"""
You are a medical assistant. Simplify clinical notes into plain English for patients.

Example 1:
Clinical Note: "The patient was diagnosed with pneumonia based on chest X-ray showing lung infiltrates."
Simplified Version: "You have a lung infection called pneumonia. It was found on your chest X-ray."

Example 2:
Clinical Note: "The patient is experiencing hyperglycemia secondary to poorly controlled diabetes mellitus type II."
Simplified Version: "Your blood sugar is too high because your diabetes hasn't been well controlled."

Now simplify this note:

Clinical Note:
\"\"\"{note}\"\"\"

Simplified Version:
"""

# Generate the prompt
prompt = few_shot_prompt(note_text)

# Send to DeepSeek model via Ollama
response = ollama.chat(
    model='deepseek-r1:8b',
    messages=[
        {'role': 'user', 'content': prompt}
    ]
)

# Print the result
print("\n🧠 DeepSeek Model Response:")
print(response['message']['content'])

# Close DB
conn.close()



🧠 DeepSeek Model Response:
<think>
Okay, so I need to simplify this clinical note into plain English for the patient. Let me read through it carefully.

First, the patient was admitted on September 16, 2018, and discharged on September 29, 2018. His birthday is February 27, 2007, and he's male. He was in the CCU service.

There's an addendum saying the summary covers from September 20 to September 29, 2018.

Let me go through each point one by one.

1. Gastrointestinal bleeding: The patient had worsening pain in his throat and esophagus and vomited blood (500cc). He was evaluated by GI and had an EGD, which showed ischemic changes in the esophagus. He was intubated for the EGD and moved to the ICU. He started on Pantoprazole drip, then switched to oral Pantoprazole. No more vomiting of blood, and his throat pain resolved.

2. Renal failure: He had chronic kidney issues and symptoms of uremia before admission. During hospitalization, his creatinine spiked due to dye from a catheter, po

In [6]:
prompt_types = {
    "zero_shot": zero_shot_prompt(note_text),
    "few_shot": few_shot_prompt(note_text),
    "chain_of_thought": chain_of_thought_prompt(note_text),
    "reasoning_and_diagnosis": reasoning_and_diagnosis_prompt(note_text)
}

for prompt_name, prompt_content in prompt_types.items():
    print(f"\n=== 🧪 Running {prompt_name} ===")
    response = ollama.chat(
        model='deepseek-r1:8b',
        messages=[
            {'role': 'user', 'content': prompt_content}
        ]
    )
    print(f"{prompt_name} Output:\n", response['message']['content'])



=== 🧪 Running zero_shot ===
zero_shot Output:
 <think>
Okay, I need to help the user by simplifying a clinical note into plain English. The user is a medical assistant, so they probably deal with this kind of task regularly. Let me read through the query again.

The clinical note has a lot of medical jargon and dates. The patient is admitted on 9/16/2182 and discharged on 9/29/2182. His date of birth is 2/27/2107, and he's male. He's in the CCU service. There's an addendum noting the hospital stay from 9/20 to 9/29.

Looking at the sections, there are four main points: Gastrointestinal Bleed, Renal Failure, Acute Coronary Syndrome, and Infectious Disease. Each has detailed information that needs to be simplified.

Starting with GI bleed: He vomited blood, had esophagitis likely due to ischemia. He was intubated for EGD, transferred to ICU, started on Pantoprazole drip, then switched to oral. No more bleeding and discomfort resolved.

Renal Failure: He had chronic kidney issues and acu

In [17]:
import json
import re

def load_note(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)
    return data

def extract_label(note_text):
    # Extract discharge diagnosis from the note text
    match = re.search(r"Discharge Diagnosis:\s*(.+)", note_text)
    if match:
        diagnosis = match.group(1).strip()
        return diagnosis
    else:
        return "Unknown"

# Load your single note
note_data = load_note('./generated_prompts/note_64.json')

# Extract the input and label
clinical_note = note_data["original_note"]
diagnosis = extract_label(clinical_note)

print(f"✅ Extracted Diagnosis: {diagnosis}")


✅ Extracted Diagnosis: Acute Coronary Syndrome


In [19]:
import os
import json
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

def extract_label(note_text):
    """Extracts the discharge diagnosis from the note text."""
    match = re.search(r"Discharge Diagnosis:\s*(.+)", note_text, re.IGNORECASE)
    return match.group(1).strip() if match else "Unknown"

def load_notes_from_directory(folder_path):
    """Loads all notes and diagnoses from a directory of JSON files."""
    texts = []
    labels = []
    
    for filename in os.listdir(folder_path):
        if filename.endswith('.json'):
            with open(os.path.join(folder_path, filename), 'r') as f:
                note = json.load(f)
                note_text = note.get('original_note', '').strip()
                label = extract_label(note_text)
                
                if label != "Unknown":  # Skip notes without clear diagnosis
                    texts.append(note_text)
                    labels.append(label)
    
    return texts, labels

def train_and_evaluate_classifier(texts, labels):
    """Splits the data, trains the classifier, and prints evaluation results."""
    X_train, X_test, y_train, y_test = train_test_split(
        texts, labels, test_size=0.2, random_state=42
    )

    model = make_pipeline(
        TfidfVectorizer(stop_words='english', max_features=10000),
        MultinomialNB()
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    print("\n✅ Classification Report:")
    print(classification_report(y_test, predictions))

    return model

# === Usage ===

folder_path = './generated_prompts'  # update this!
texts, labels = load_notes_from_directory(folder_path)

if texts:
    trained_model = train_and_evaluate_classifier(texts, labels)
else:
    print("⚠️ No valid notes found with discharge diagnoses.")



✅ Classification Report:
                                         precision    recall  f1-score   support

                Acute Coronary Syndrome       0.00      0.00      0.00       0.0
Left hip osteomyelitis/septic arthritis       0.00      0.00      0.00       1.0

                               accuracy                           0.00       1.0
                              macro avg       0.00      0.00      0.00       1.0
                           weighted avg       0.00      0.00      0.00       1.0



/opt/homebrew/Caskroom/miniconda/base/envs/deepseek_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/homebrew/Caskroom/miniconda/base/envs/deepseek_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/homebrew/Caskroom/miniconda/base/envs/deepseek_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behav

In [24]:
new_note = "Patient admitted to CCU for chest pain, diagnosed with Acute Coronary Syndrome..."
predicted = trained_model.predict([new_note])[0]
print(f"Predicted Diagnosis: {predicted}")


Predicted Diagnosis: Acute Coronary Syndrome
